1. 라이브러리 임포트

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input

2. 데이터 불러오기

In [28]:
df = pd.read_csv("scaled.csv")
print(f"원본 데이터 shape: {df.shape}")

원본 데이터 shape: (284807, 31)


3. 'Class' 결측치 확인 및 제거

In [29]:
# 3-1. (확인) 'Class' 열에 NaN이 있는지 확인
nan_count = df['Class'].isnull().sum()
if nan_count > 0:
    print(f"경고: 'Class' 열에 {nan_count}개의 NaN(결측치)이 있습니다. 이 행들을 제거합니다.")
    # 3-2. (해결) 'Class' 열에 NaN이 있는 행(row)을 제거
    df.dropna(subset=['Class'], inplace=True)

4. 입력(X), 출력(y) 분리

In [30]:
X = df.drop(columns=['Class'])
Y = df['Class']

# 이미 scaled.csv는 스케일링 완료된 상태이므로 StandardScaler 불필요

5. Train/Test Split

In [32]:
train_X, test_X, train_Y, test_Y = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)
print("Train/Test 분리 완료")

Train/Test 분리 완료


6. 클래스 가중치 계산

In [33]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_Y),
    y=train_Y
)
class_weights = {0: class_weights[0], 1: class_weights[1]}
print(f"클래스 가중치: {class_weights}")

클래스 가중치: {0: np.float64(0.5008661206149896), 1: np.float64(289.14340101522845)}


7. MLP 모델 정의

In [36]:
def mlp(inputDim):
    model = Sequential([
        Input(shape=(inputDim,)),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),

        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy'
    )
    return model

8. 모델 학습

In [41]:
model = mlp(train_X.shape[1])
history =model.fit(train_X, train_Y,
          class_weight=class_weights,
          verbose=1,
          validation_split=0.2,
          batch_size=512,
          epochs=8)

Epoch 1/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - loss: 0.2942 - val_loss: 0.1607
Epoch 2/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1953 - val_loss: 0.1153
Epoch 3/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1710 - val_loss: 0.0898
Epoch 4/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1237 - val_loss: 0.1005
Epoch 5/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1327 - val_loss: 0.0869
Epoch 6/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.1042 - val_loss: 0.0816
Epoch 7/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1008 - val_loss: 0.0597
Epoch 8/8
357/357 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0965 - val_loss: 0.0820


9. 예측 및 평가

In [43]:
from sklearn.metrics import precision_score, recall_score

Y_predicted_probability = model.predict(test_X)
threshold = 0.5
Y_predict_value = (Y_predicted_probability  > threshold).astype(int)

recall = recall_score(test_Y, Y_predict_value )
precision = precision_score(test_Y, Y_predict_value )

print("\n모델 평가 결과 (MLP, threshold=0.5)")
print(f"Threshold = {threshold}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 896us/step

모델 평가 결과 (MLP, threshold=0.5)
Threshold = 0.5
Recall: 0.9082
Precision: 0.0531
